# Enterprise CLV Forecaster - Lightweight Training
This notebook tunes a LightGBM model using Optuna and exports a lightweight `.pkl` artifact.

In [1]:
!pip install -q optuna lightgbm s3fs scikit-learn pandas numpy joblib

In [2]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import gc

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [5]:
print("Downloading Data from S3 with Memory Downcasting...")
schema = {
    'customer_id': 'string',
    'recency': 'int32',
    'model_recency': 'int32',
    'customer_tenure': 'int32',
    'frequency': 'int32',
    'monetary': 'float32',
    'avg_order_value': 'float32',
    'target_clv': 'float32'
}

df = pd.read_csv('s3://amzn-s3-hm-clv/hm_data/aggregated_clv.csv', dtype=schema)

In [6]:
features = ['recency', 'model_recency', 'customer_tenure', 'frequency', 'monetary', 'avg_order_value']
X = df[features]
y = df['target_clv']

del df 
gc.collect()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    
    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, y_train)
    preds = np.clip(model.predict(X_test), 0.0, None)
    return mean_absolute_error(y_test, preds)

print("Starting Memory-Safe Optuna Optimization...")
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15, gc_after_trial=True)

best_params = study.best_params
best_params['random_state'] = 42
best_params['n_jobs'] = -1
best_params['verbose'] = -1  

Starting Memory-Safe Optuna Optimization...


In [10]:
study.best_score

AttributeError: 'Study' object has no attribute 'best_score'

In [9]:
print("Training Champion Model...")
champion = lgb.LGBMRegressor(**best_params)
champion.fit(X_train, y_train)

joblib.dump(champion, 'clv_model.pkl')
print("✅ Model successfully optimized and saved as 'clv_model.pkl'!")

Training Champion Model...
✅ Model successfully optimized and saved as 'clv_model.pkl'!
